# Práctica 1: Perceptrón multicapa.

Tu jefe pidió a RH que recolectara datos de desempeño de tus compañeros, los resultados se almacenaron en un csv. El punto critico de estos datos es la satisfacción del empleado, entonces ¿Podremos estimar la satisfacción de los empleados con los datos recabados?.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
import numpy as np
from tensorflow.keras import layers, models


df = pd.read_csv('Extended_Employee_Performance_and_Productivity_Data.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 20 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   Employee_ID                  100000 non-null  int64  
 1   Department                   100000 non-null  str    
 2   Gender                       100000 non-null  str    
 3   Age                          100000 non-null  int64  
 4   Job_Title                    100000 non-null  str    
 5   Hire_Date                    100000 non-null  str    
 6   Years_At_Company             100000 non-null  int64  
 7   Education_Level              100000 non-null  str    
 8   Performance_Score            100000 non-null  int64  
 9   Monthly_Salary               100000 non-null  float64
 10  Work_Hours_Per_Week          100000 non-null  int64  
 11  Projects_Handled             100000 non-null  int64  
 12  Overtime_Hours               100000 non-null  int64  
 13  Sick_Days  

In [2]:
# Filtrar las columnas numéricas
numeric_columns = df.select_dtypes(include=['number']).drop('Employee_ID',axis=1)


# Si numeric_columns es un Index, conviértelo a lista
cols = list(numeric_columns)

fig, axes = plt.subplots(1, len(cols), figsize=(5 * len(cols), 4))

for i, col in enumerate(cols):
    axes[i].hist(df[col], bins=20, color='skyblue', edgecolor='black')
    axes[i].set_title(col)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

C:\Users\Agadez\AppData\Local\Temp\ipykernel_2836\758830218.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Problemas**, tenemos distribuciones con picos, esos nos indica categorías. Por otro lado, tenemos variables con "valles" en su distribución (distribuciones multimodales) por lo que resultaría óptimo aplicar técnicas de feature engeneering. Por último tenemos distribuciones uniformes, por lo que cada una requeriría un procesamiento indivudual, hagamos la vista gorda e intentemos ajustar un MLP con estos datos, solo estandaricemos nuestros datos.

---

## Implementación de Red:

To**memos los datos numéricos como nuestra variable X, y la variable objetivo como ***'Employee_Satisfaction_Score'***.
- **Actividad 1**: Para todos los strings ``'@modif@'`` que aparescan en el siguiente bloque de código cámbialos para que el código funcione.

In [3]:
X = numeric_columns.drop('Employee_Satisfaction_Score',axis = 1)
y = numeric_columns['Employee_Satisfaction_Score']
y = y.apply(lambda x: round(x)-1) #Cambiamos la variable objetivo a 5 categorías numéricas

scaler = StandardScaler()
X_standar = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_standar, y, test_size=0.33, random_state=42)

y_onehot_train = tf.keras.utils.to_categorical(y_train, 5)
y_onehot_test = tf.keras.utils.to_categorical(y_test,5)

- **Actividad 2:** Implementa 3 arquitecturas de MLP, cada una con su propio nombre, cambiando la estructura de dichas arquitecturas (capas, neuronas por capa, función de activación, etc). 

In [4]:
# En las siguientes celdas, construye tu modelo con tensorflow.keras solo la arquitectura
input_dim = X_train.shape[1]

# Arquitectura 1: MLP poco profundo
mlp_shallow = models.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(32, activation='relu'),
    layers.Dense(16, activation='relu'),
    layers.Dense(5, activation='softmax')
], name='mlp_shallow')

# Arquitectura 2: MLP más profundo con dropout
mlp_deep = models.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(32, activation='tanh'),
    layers.Dense(5, activation='softmax')
], name='mlp_deep')

# Arquitectura 3: MLP ancho con activación elu
mlp_wide = models.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(256, activation='elu'),
    layers.Dense(128, activation='elu'),
    layers.Dense(5, activation='softmax')
], name='mlp_wide')

mlp_shallow.summary()
mlp_deep.summary()
mlp_wide.summary()

Model: "mlp_shallow"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 5)              │            85 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,029 (4.02 KB)

 Trainable params: 1,029 (4.02 KB)

 Non-trainable params: 0 (0.00 B)

...

Model: "mlp_wide"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_7 (Dense)                 │ (None, 256)            │         3,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 36,869 (144.02 KB)

 Trainable params: 36,869 (144.02 KB)

 Non-trainable params: 0 (0.00 B)

- **Actividad 3:** Compila y ajusta tus tres modelos con sus respectivos hiperparámetros.

In [5]:
# En las siguientes celdas, compila y entrena el modelo
mlp_shallow.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
mlp_deep.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
mlp_wide.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history_shallow = mlp_shallow.fit(X_train, y_onehot_train, validation_data=(X_test, y_onehot_test), epochs=10, batch_size=64, verbose=1)
history_deep = mlp_deep.fit(X_train, y_onehot_train, validation_data=(X_test, y_onehot_test), epochs=10, batch_size=64, verbose=1)
history_wide = mlp_wide.fit(X_train, y_onehot_train, validation_data=(X_test, y_onehot_test), epochs=10, batch_size=128, verbose=1)

print('shallow eval:', mlp_shallow.evaluate(X_test, y_onehot_test, verbose=0))
print('deep eval:', mlp_deep.evaluate(X_test, y_onehot_test, verbose=0))
print('wide eval:', mlp_wide.evaluate(X_test, y_onehot_test, verbose=0))

Epoch 1/10


   1/1047 ━━━━━━━━━━━━━━━━━━━━ 16:01 919ms/step - accuracy: 0.1875 - loss: 1.6310

  52/1047 ━━━━━━━━━━━━━━━━━━━━ 0s 994us/step - accuracy: 0.2335 - loss: 1.6224   

 109/1047 ━━━━━━━━━━━━━━━━━━━━ 0s 933us/step - accuracy: 0.2451 - loss: 1.5971

 167/1047 ━━━━━━━━━━━━━━━━━━━━ 0s 912us/step - accuracy: 0.2506 - loss: 1.5874

...

483/524 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.2566 - loss: 1.5562

524/524 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.2563 - loss: 1.5559 - val_accuracy: 0.2494 - val_loss: 1.5586


shallow eval: [1.5589667558670044, 0.2507878839969635]


deep eval: [1.5573838949203491, 0.2526363730430603]


wide eval: [1.5586131811141968, 0.24936363101005554]


- **Actividad 4:** Sube tus cambios al repositorio, envía el link de tu repositorio a la actividad 2 de tu checkpoint 2 y contesta las preguntas de dicha actividad.